# Project · ScamGuard AI — scam-message detector

A 3-step pipeline (**build prompt → call LLM → parse JSON**) plus an evaluation script that measures accuracy, recall and F1 on a labelled dataset.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai pandas pydantic python-dotenv tqdm

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

## 1 · Create the project files in this Colab session
Each cell below writes one file of the original project (`%%writefile`). Read them as you go — it's the same code as in the app's project page.

In [ ]:
%cd /content
!mkdir -p scamguard_ai
%cd scamguard_ai

In [ ]:
import os
for d in ['llm', 'llm/prompts', 'pipeline', 'pipeline/scam_detector']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile utils.py
import logging
import re
import json
from pathlib import Path

def get_logger(name: str) -> logging.Logger:
    """Get a simple logger for the given name."""
    logging.basicConfig(
        level=logging.INFO,
        format='[%(asctime)s] %(levelname)s: %(message)s'
    )
    return logging.getLogger(name)

def extract_json_from_text(text: str) -> dict:
    """Extract JSON from text string. Returns empty dict if not found."""
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}
    except json.JSONDecodeError:
        return {}

def load_file(file_path: str) -> str:
    """Load and return file contents as string."""
    return Path(file_path).read_text().strip()

In [ ]:
%%writefile config.py
"""
Minimal configuration file for the Scam Detection project.
Contains only the essential settings used across the project.
"""

import os
from pathlib import Path
from dotenv import load_dotenv

# Get project root directory
PROJECT_ROOT = Path(__file__).parent

# Load environment variables from .env file
load_dotenv(PROJECT_ROOT / ".env")

# API Configuration
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")  # from .env / Colab Secrets - never hard-code

# LLM Settings
DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")  # e.g. gemini-3.5-flash-lite
MAX_RETRIES = 3
RETRY_DELAY = 2

# Dataset Configuration
DEFAULT_DATASET = PROJECT_ROOT / "scam_detection_dataset.csv"
TEST_DATASET = PROJECT_ROOT / "test_scam_dataset.csv"

# Text column names to look for in datasets
TEXT_COLUMNS = ["text", "message_text", "message"]
LABEL_COLUMN = "label"

# Paths
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
LOGS_DIR = PROJECT_ROOT / "logs"

# Create output directories if they don't exist
OUTPUTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

def get_dataset_path(filename: str) -> Path:
    """
    Find dataset file in project directory.

    Args:
        filename: Name of the dataset file

    Returns:
        Path to the dataset file

    Raises:
        FileNotFoundError: If file not found
    """
    # Try direct path first
    if Path(filename).exists():
        return Path(filename)

    # Try project root
    project_path = PROJECT_ROOT / filename
    if project_path.exists():
        return project_path

    raise FileNotFoundError(f"Dataset '{filename}' not found")

In [ ]:
%%writefile llm/__init__.py
# llm/__init__.py

# from .client import LLMClient
# from .prompts import generate_zero_shot_prompt, generate_few_shot_prompt, REACT_PROMPT_TEMPLATE
# from .parser import ScamDetectionOutput, validate_output

# __all__ = [
#     'LLMClient',
#     'generate_zero_shot_prompt', 
#     'generate_few_shot_prompt',
#     'REACT_PROMPT_TEMPLATE',
#     'ScamDetectionOutput',
#     'validate_output'
# ]

In [ ]:
%%writefile llm/client.py
"""LLM Client for Google Gemini API with retry logic."""

import time
from google import genai
from config import GEMINI_API_KEY, DEFAULT_MODEL, MAX_RETRIES, RETRY_DELAY
from utils import get_logger

logger = get_logger(__name__)


class LLMClient:
    """Gemini API client"""

    def __init__(self, model_name=DEFAULT_MODEL, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY):
        self.model_name = model_name
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.client = genai.Client(api_key=GEMINI_API_KEY)

    def call(self, prompt: str, **kwargs) -> str:
        """Send prompt to Gemini API"""
        for attempt in range(self.max_retries + 1):
            try:
                response = self.client.models.generate_content(
                    contents=prompt, 
                    model=self.model_name, 
                    **kwargs
                )
                if response and response.text:
                    return response.text.strip()
                else:
                    raise Exception("Empty response received")

            except Exception as e:
                if attempt == self.max_retries:
                    raise Exception(f"API call failed after {self.max_retries + 1} attempts: {e}")

                time.sleep(self.retry_delay * (2 ** attempt))

In [ ]:
%%writefile llm/parser.py
# llm/parser.py

from pydantic import BaseModel, Field, ValidationError
from typing import List

class ScamDetectionOutput(BaseModel):
    label: str = Field(..., description="Scam | Not Scam | Uncertain")
    reasoning: str = Field(..., description="Step-by-step analysis of why the label was assigned")
    intent: str = Field(..., description="Short description of the user's intent")
    risk_factors: List[str] = Field(..., description="List of red flags identified in the message")

def validate_output(response: dict) -> ScamDetectionOutput:
    """
    Validate and return structured ScamDetectionOutput model.

    Args:
        llm_response: Dictionary containing the LLM response data

    Returns:
        Validated ScamDetectionOutput instance

    Raises:
        ValueError: If validation fails with details about what went wrong
    """
    try:
        return ScamDetectionOutput(**response)
    except ValidationError as e:
        raise ValueError(f"LLM output failed schema validation: {str(e)}")

In [ ]:
%%writefile llm/validator.py
# llm/validator.py

from pydantic import BaseModel, Field, ValidationError
from typing import List

class ScamDetectionOutput(BaseModel):
    label: str = Field(..., description="Scam | Not Scam | Uncertain")
    reasoning: str = Field(..., description="Step-by-step analysis of why the label was assigned")
    intent: str = Field(..., description="Short description of the user's intent")
    risk_factors: List[str] = Field(..., description="List of red flags identified in the message")

def validate_output(response: dict) -> ScamDetectionOutput:
    """
    Validate and return structured ScamDetectionOutput model.
    If validation fails, raise a clear error.
    """
    try:
        return ScamDetectionOutput(**response)
    except ValidationError as e:
        raise ValueError(f"LLM output failed schema validation: {str(e)}")

In [ ]:
%%writefile llm/prompts.py
# llm/prompts.py

from pathlib import Path
from utils import load_file

# Directory paths
PROMPTS_DIR = Path(__file__).parent / "prompts"

# Load prompt templates from external files
def load_prompt(filename: str) -> str:
    """Load a prompt template from file."""
    return load_file(PROMPTS_DIR / filename)

# Available prompt templates
PROMPT = load_prompt("react.md") # <-- Edit this to use different prompts

def generate_prompt(user_input: str) -> str:
    """
    Generate a prompt for scam detection - choose your strategy here!

    Available strategies:
    - REACT_PROMPT: Step-by-step reasoning 
    - FEWSHOT_PROMPT: Learning from examples
    - STRICT_JSON_PROMPT: Clean JSON output
    - SIMPLIFIED_PROMPT: Quick classification

    Args:
        user_input: The message to analyze
        use_react: Legacy parameter, defaults to ReAct strategy

    Returns:
        A formatted prompt string ready for LLM execution
    """
    template = PROMPT
    return f"{template}\n\nUser Message:\n{user_input.strip()}"

In [ ]:
%%writefile llm/prompts/react.md
You are a highly reliable and safety-focused AI system trained to identify potentially scammy, manipulative, or deceptive intent in text-based communication.

Follow this exact structured reasoning format for each message:

1. **Thought**: Analyze the tone, language, urgency, and phrasing patterns
2. **Action**: Classify if this is likely a scam or not based on evidence
3. **Observation**: Justify your classification with specific details
4. **Final Answer**: Output a structured JSON with the following fields:

```json
{
  "label": "Scam | Not Scam | Uncertain",
  "reasoning": "<step-by-step analysis>",
  "intent": "<short description of user intent>",
  "risk_factors": ["<e.g., urgency, financial request, impersonation>"]
}
```

**Key Analysis Points:**

- Urgency tactics (limited time offers, immediate action required)
- Financial requests (money, bank details, card info)
- Impersonation (claiming to be from legitimate organizations)
- Suspicious links or attachments
- Grammar and spelling quality
- Emotional manipulation tactics

Be cautious when unsure. Do not make up details beyond the input.

In [ ]:
%%writefile llm/prompts/fewshot.md
# Scam Detection - Few-Shot Learning

You are an expert at identifying scams in text messages. Learn from these examples and apply the same reasoning to new messages.

## Example 1:
**Message:** "Congratulations! You've won a free iPhone. Click this link and enter your bank details to claim."

**Analysis:**
```json
{
  "label": "Scam",
  "reasoning": "This message uses classic scam tactics: unexpected reward, request for sensitive financial information, and urgency through a link click requirement.",
  "intent": "Trick user into sharing banking information",
  "risk_factors": ["Unexpected reward", "Link click", "Bank details request"]
}
```

## Example 2:
**Message:** "Dear customer, your bill is due. Please visit our portal and pay by 5th."

**Analysis:**
```json
{
  "label": "Not Scam",
  "reasoning": "This is a standard billing reminder with no threatening or suspicious tone. No request for sensitive info outside normal billing process.",
  "intent": "Inform about billing and prompt payment",
  "risk_factors": []
}
```

## Example 3:
**Message:** "URGENT: Your account will be suspended! Call 555-FAKE immediately or lose access forever!"

**Analysis:**
```json
{
  "label": "Scam",
  "reasoning": "Uses high urgency, threatens account suspension, and provides suspicious phone number. Legitimate companies rarely use such aggressive tactics.",
  "intent": "Create panic to force immediate action",
  "risk_factors": ["High urgency", "Threats", "Suspicious phone number", "Fear tactics"]
}
```

## Your Task:
Analyze the following message using the same structured approach. Output only the JSON response.

In [ ]:
%%writefile llm/prompts/strict_json.txt
SCAM DETECTION SYSTEM - STRICT JSON MODE

Analyze the provided text message and classify it as either "Scam", "Not Scam", or "Uncertain".

OUTPUT REQUIREMENTS:
- Respond with ONLY valid JSON
- No additional text, explanations, or formatting outside the JSON
- Use exact field names as specified
- Ensure all strings are properly escaped

JSON SCHEMA:
{
  "label": "Scam | Not Scam | Uncertain",
  "reasoning": "Brief explanation of classification decision",
  "intent": "What the sender is trying to achieve",
  "risk_factors": ["array", "of", "risk", "factors"]
}

CLASSIFICATION CRITERIA:
- Scam: Contains deceptive, manipulative, or fraudulent intent
- Not Scam: Legitimate communication with no malicious intent  
- Uncertain: Ambiguous content that cannot be clearly classified

RISK FACTORS TO IDENTIFY:
- Urgency tactics
- Financial requests
- Impersonation
- Suspicious links
- Emotional manipulation
- Poor grammar/spelling
- Threats or pressure

MESSAGE TO ANALYZE:

In [ ]:
%%writefile llm/prompts/simplified.txt
Simple Scam Detection

Is this message a scam? Look for:
✓ Requests for money/personal info
✓ Urgent threats or pressure
✓ Too-good-to-be-true offers
✓ Impersonation of legitimate services

Classify as:
- Scam: Clearly fraudulent or deceptive
- Not Scam: Appears legitimate 
- Uncertain: Cannot determine clearly

Respond in this format:
{
  "label": "Scam/Not Scam/Uncertain",
  "reasoning": "Why you classified it this way",
  "intent": "What the sender wants",
  "risk_factors": ["list of concerns"]
}

In [ ]:
%%writefile llm/prompts/few_shot_examples.txt
# Few-shot Example 1
Message:
Congratulations! You've won a free iPhone. Click this link and enter your bank details to claim.

Expected Response:
{
  "label": "Scam",
  "reasoning": "The message uses reward bait and requests sensitive info.",
  "intent": "Trick user into sharing banking information",
  "risk_factors": ["Unexpected reward", "Link click", "Bank details"]
}

---
# Few-shot Example 2
Message:
Dear customer, your bill is due. Please visit our portal and pay by 5th.

Expected Response:
{
  "label": "Not Scam",
  "reasoning": "This is a standard billing reminder with no threatening or suspicious tone.",
  "intent": "Inform about billing and prompt payment",
  "risk_factors": []
}

---
# Few-shot Example 3
Message:
URGENT: Your account will be suspended! Call 555-FAKE immediately or lose access forever!

Expected Response:
{
  "label": "Scam",
  "reasoning": "Uses high urgency, threats, and suspicious phone number to create panic.",
  "intent": "Create panic to force immediate action",
  "risk_factors": ["High urgency", "Threats", "Suspicious phone number", "Fear tactics"]
}

---
# Few-shot Example 4
Message:
Your package from Amazon is delayed. Track it here: bit.ly/track123

Expected Response:
{
  "label": "Uncertain",
  "reasoning": "Could be legitimate package notification but uses shortened URL which is risky.",
  "intent": "Either provide tracking info or phish for information",
  "risk_factors": ["Shortened URL", "Potential impersonation"]
}

---

In [ ]:
%%writefile pipeline/__init__.py

In [ ]:
%%writefile pipeline/scam_detector/__init__.py
# # pipeline/scam_detector/__init__.py

# from .detector import ScamDetector

# __all__ = ['ScamDetector']

In [ ]:
%%writefile pipeline/scam_detector/builder.py
"""Prompt building functions for scam detection."""

from llm.prompts import generate_prompt


def build_prompt(message: str, strategy: str = "react") -> str:
    """
    Build a prompt for the given message based on strategy.

    Args:
        message: The input message to create a prompt for
        strategy: The prompting strategy to use

    Returns:
        A formatted prompt string ready for LLM execution
    """
    if strategy == "react":
        return generate_prompt(message)
    else:
        raise NotImplementedError(f"Strategy '{strategy}' is not supported yet.")

In [ ]:
%%writefile pipeline/scam_detector/executor.py
# pipeline/scam_detector/executor.py

from typing import Optional
from llm.client import LLMClient
from utils import get_logger

logger = get_logger(__name__)

class LLMExecutor:
    """Executes prompts using the LLM client."""

    def __init__(self, model: Optional[str] = None) -> None:
        """
        Initialize the LLM executor.

        Args:
            model: Optional specific model to use. If None, uses default from config.
        """
        self.llm: LLMClient = LLMClient(model) if model else LLMClient()
        logger.info("Initialized LLMExecutor")

    def execute(self, prompt: str) -> str:
        """
        Execute the prompt using the LLM and return the raw response.

        Args:
            prompt: The formatted prompt string to send to the LLM

        Returns:
            Raw text response from the LLM

        Raises:
            Exception: If LLM call fails after all retries
        """
        logger.info(f"Executing LLM with prompt length: {len(prompt)}")
        try:
            response = self.llm.call(prompt)
            logger.info(f"LLM execution successful, response length: {len(response)}")
            return response
        except Exception as e:
            logger.error(f"LLM execution failed: {str(e)}")
            raise

In [ ]:
%%writefile pipeline/scam_detector/parser.py
# pipeline/scam_detector/parser.py

from typing import Dict, Any
from utils import get_logger, extract_json_from_text

logger = get_logger(__name__)

class OutputParser:
    """Parses LLM output into structured format."""

    def parse_llm_output(self, llm_output: str) -> Dict[str, Any]:
        """
        Extract and parse JSON structure from LLM response.

        Args:
            llm_output: Raw text output from the LLM

        Returns:
            Dictionary containing structured detection results with keys:
            - label: str - Classification result ("Scam", "Not Scam", "Uncertain")
            - reasoning: str - Step-by-step analysis
            - intent: str - Description of user intent
            - risk_factors: List[str] - List of identified red flags

        Note:
            If parsing fails, returns a fallback dictionary with "Uncertain" label
            and error information in the reasoning field.
        """
        logger.info(f"Parsing LLM output of length: {len(llm_output)}")

        # Try to extract JSON using utils function
        parsed_json = extract_json_from_text(llm_output)

        if parsed_json:
            logger.info("Successfully parsed LLM output to JSON.")
            return parsed_json
        else:
            logger.warning("No JSON found in LLM output.")
            # Return fallback result
            fallback_result = {
                "label": "Uncertain",
                "reasoning": "Failed to parse response: No JSON found",
                "intent": "Could not determine",
                "risk_factors": []
            }
            return fallback_result

In [ ]:
%%writefile pipeline/scam_detector/detector.py
# pipeline/scam_detector/detector.py

from typing import List, Dict, Any
from .builder import build_prompt
from .executor import LLMExecutor
from .parser import OutputParser
from utils import get_logger

logger = get_logger(__name__)

class ScamDetector:
    """Orchestrates the scam detection pipeline."""

    def __init__(self, strategy: str = "react") -> None:
        """Initializes the pipeline components."""
        self.executor = LLMExecutor()
        self.parser = OutputParser()
        self.strategy = strategy
        logger.info(f"Initialized ScamDetector with strategy: {self.strategy}")

    def detect(self, message: str) -> Dict[str, Any]:
        """Runs scam detection on a single message."""
        logger.info(f"Starting detection for message length: {len(message)}")
        try:
            # The 3-step pipeline
            prompt = build_prompt(message, self.strategy)
            raw_response = self.executor.execute(prompt)
            parsed_result = self.parser.parse_llm_output(raw_response)

            logger.info(f"Detection successful. Result: {parsed_result.get('label', 'Unknown')}")
            return parsed_result

        except Exception as e:
            logger.error(f"Detection pipeline failed: {e}")
            # Re-raise the exception to be handled by the caller (UI layer)
            raise

    def detect_batch(self, messages: List[str]) -> List[Dict[str, Any]]:
        """Runs scam detection on a list of messages."""
        total_messages = len(messages)
        logger.info(f"Starting batch detection for {total_messages} messages.")

        results: List[Dict[str, Any]] = []

        for i, message in enumerate(messages):
            try:
                result = self.detect(message)
                results.append(result)
            except Exception as e:
                logger.warning(f"Failed to process message {i+1}/{total_messages}: {e}")
                # Append a fallback error result for the failed message
                error_result = {
                    "label": "Uncertain",
                    "reasoning": f"Error processing message: {e}",
                    "intent": "Could not determine",
                    "risk_factors": ["processing_error"]
                }
                results.append(error_result)

        successful = sum(1 for r in results if r.get("label") != "Uncertain" or "processing_error" not in r.get("risk_factors", []))
        logger.info(f"Batch processing complete. {successful}/{total_messages} succeeded.")
        return results

In [ ]:
%%writefile evaluate.py
# evaluate.py

import pandas as pd
from tqdm import tqdm
import argparse

from pipeline.scam_detector.detector import ScamDetector

def calculate_metrics(actual_labels, predicted_labels):
    """Calculate accuracy, recall, and F1-score for scam detection."""
    total = len(actual_labels)
    if total == 0:
        return {"total": 0, "correct": 0, "accuracy": 0.0, "class_metrics": {}}

    # Clean up labels (remove spaces, make lowercase)
    actual_clean = [label.lower().strip() for label in actual_labels]
    predicted_clean = [label.lower().strip() for label in predicted_labels]

    # Calculate overall accuracy
    correct = 0
    for actual, pred in zip(actual_clean, predicted_clean):
        if actual == pred:
            correct += 1

    accuracy = (correct / total) * 100

    # Calculate metrics for each class
    class_metrics = {}
    unique_labels = set(actual_clean)

    for label in unique_labels:
        # Count correct and incorrect predictions for this class
        true_positives = 0
        false_negatives = 0
        false_positives = 0

        for actual, pred in zip(actual_clean, predicted_clean):
            if actual == label and pred == label:
                true_positives += 1
            elif actual == label and pred != label:
                false_negatives += 1
            elif actual != label and pred == label:
                false_positives += 1

        # Calculate metrics
        if true_positives + false_negatives > 0:
            recall = (true_positives / (true_positives + false_negatives)) * 100
        else:
            recall = 0

        if true_positives + false_positives > 0:
            precision = (true_positives / (true_positives + false_positives)) * 100
        else:
            precision = 0

        if precision + recall > 0:
            f1_score = (2 * precision * recall) / (precision + recall)
        else:
            f1_score = 0

        class_metrics[label] = {
            "recall": round(recall, 2),
            "f1_score": round(f1_score, 2)
        }

    return {
        "total": total,
        "correct": correct,
        "accuracy": round(accuracy, 2),
        "class_metrics": class_metrics
    }

def evaluate_model(dataset_path, limit=None, verbose=False, batch_size=10):
    """Load dataset, run predictions, and evaluate performance."""
    # Load the dataset
    try:
        df = pd.read_csv(dataset_path)
        print(f"Loaded dataset with {len(df)} rows")
    except FileNotFoundError:
        print(f"Error: Dataset file not found at {dataset_path}")
        return

    if limit:
        df = df.head(limit)
        print(f"Limiting evaluation to the first {limit} messages.")

    # Initialize detector
    detector = ScamDetector()

    # Prepare data for batch processing
    messages = df['message_text'].tolist()
    actual_labels = df['label'].tolist()

    print(f"\nEvaluating {len(messages)} messages in batches of {batch_size}...")

    # Process messages in batches
    predicted_labels = []
    total_batches = (len(messages) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(messages), batch_size), desc="Processing batches", total=total_batches):
        batch_messages = messages[i:i + batch_size]
        batch_actual = actual_labels[i:i + batch_size]

        try:
            # Process batch
            batch_results = detector.detect_batch(batch_messages)
            batch_predicted = [result.get("label", "Uncertain") for result in batch_results]

            if verbose:
                for j, (msg, true_label, pred_label) in enumerate(zip(batch_messages, batch_actual, batch_predicted)):
                    print(f"\nMessage {i+j+1}: {msg[:50]}...")
                    print(f"  True: {true_label}")
                    print(f"  Predicted: {pred_label}")

            predicted_labels.extend(batch_predicted)

        except Exception as e:
            print(f"Error processing batch {i//batch_size + 1}: {e}")
            # Add fallback predictions for failed batch
            predicted_labels.extend(["Uncertain"] * len(batch_messages))

    print("\nEvaluation complete.")

    # Calculate and print the final metrics
    metrics = calculate_metrics(actual_labels, predicted_labels)

    print("\n" + "="*50)
    print(" MODEL EVALUATION RESULTS")
    print("="*50)
    print(f"  Total Messages: {metrics['total']}")
    print(f"  Correct Predictions: {metrics['correct']}")
    print(f"  Overall Accuracy: {metrics['accuracy']}%")
    print()

    print("  PER-CLASS METRICS:")
    for label, class_metrics in metrics['class_metrics'].items():
        print(f"    {label.upper()}:")
        print(f"      Recall: {class_metrics['recall']}%")
        print(f"      F1-Score: {class_metrics['f1_score']}%")
        print()
    print("="*50)

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Evaluate the Scam Detection System.")
    parser.add_argument("dataset", help="Path to the labeled dataset (CSV file).")
    parser.add_argument("--limit", type=int, help="Limit evaluation to the first N messages.")
    parser.add_argument("--verbose", action="store_true", help="Print detailed results for each message.")
    parser.add_argument("--batch-size", type=int, default=10, help="Batch size for processing (default: 10).")

    args = parser.parse_args()
    evaluate_model(args.dataset, args.limit, args.verbose, args.batch_size)

## 2 · Data
A 60-message sample of the course dataset is embedded so this runs anywhere. To use the **full 1,000-message file**, run the upload cell afterwards.

In [ ]:
%%writefile scam_sample.csv
message_text,label,intent_type,flag_reason
Reminder: Your gas cylinder booking is confirmed.,Not Scam,Service Reminder,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Your Amazon package has been dispatched.,Not Scam,Order Confirmation,Legitimate communication
Account password changed on 5 July at 3:45 PM.,Not Scam,Account Update,Legitimate communication
Your Aadhaar has been successfully linked with your PAN.,Not Scam,Informational Alert,Legitimate communication
Your broadband bill is due on 7 July. Pay now to avoid late fees.,Not Scam,Service Reminder,Legitimate communication
Power cut scheduled from 2 PM to 4 PM tomorrow.,Not Scam,Informational Alert,Legitimate communication
New summer collection live now at Myntra. Check it out!,Not Scam,Marketing Message,Legitimate communication
Get 40% off on your first Zomato order. Use code FIRST40.,Not Scam,Marketing Message,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
"Payment of ₹1,000 received from Paytm UPI.",Not Scam,Transactional Notification,Legitimate communication
Power cut scheduled from 2 PM to 4 PM tomorrow.,Not Scam,Informational Alert,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Your broadband bill is due on 7 July. Pay now to avoid late fees.,Not Scam,Service Reminder,Legitimate communication
"₹500 has been debited from your account. Available balance: ₹12,340.",Not Scam,Transactional Notification,Legitimate communication
Power cut scheduled from 2 PM to 4 PM tomorrow.,Not Scam,Informational Alert,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Your Amazon package has been dispatched.,Not Scam,Order Confirmation,Legitimate communication
Your broadband bill is due on 7 July. Pay now to avoid late fees.,Not Scam,Service Reminder,Legitimate communication
Your Aadhaar has been successfully linked with your PAN.,Not Scam,Informational Alert,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Get 40% off on your first Zomato order. Use code FIRST40.,Not Scam,Marketing Message,Legitimate communication
Order #12456 has been shipped and will be delivered by 6 July.,Not Scam,Order Confirmation,Legitimate communication
"₹500 has been debited from your account. Available balance: ₹12,340.",Not Scam,Transactional Notification,Legitimate communication
"Payment of ₹1,000 received from Paytm UPI.",Not Scam,Transactional Notification,Legitimate communication
Your Aadhaar has been successfully linked with your PAN.,Not Scam,Informational Alert,Legitimate communication
Your Aadhaar has been successfully linked with your PAN.,Not Scam,Informational Alert,Legitimate communication
Reminder: Your gas cylinder booking is confirmed.,Not Scam,Service Reminder,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Your email has been successfully updated.,Not Scam,Account Update,Legitimate communication
Act now! Your bank account is under threat.,Scam,Urgency,Identified as Urgency
We have restricted your UPI services. Update now.,Scam,Account Suspension,Identified as Account Suspension
Get instant loan without credit check. Click here.,Scam,Loan Scam,Identified as Loan Scam
Your SIM card will be deactivated today. Verify immediately.,Scam,Fear Tactics,Identified as Fear Tactics
Your SIM card will be deactivated today. Verify immediately.,Scam,Fear Tactics,Identified as Fear Tactics
Police case filed against your name. Pay ₹5000 to stop it.,Scam,Fear Tactics,Identified as Fear Tactics
Government Grant Approved. Send your Aadhar number now.,Scam,Fake Authority,Identified as Fake Authority
Urgent! Your account will be blocked in 24 hours. Click here to verify.,Scam,Urgency,Identified as Urgency
Unusual activity detected. Please login immediately at scamurl.com.,Scam,Phishing,Identified as Phishing
"Pre-approved loan of ₹5,00,000. Apply instantly with zero documents.",Scam,Loan Scam,Identified as Loan Scam
Get instant loan without credit check. Click here.,Scam,Loan Scam,Identified as Loan Scam
Your SIM card will be deactivated today. Verify immediately.,Scam,Fear Tactics,Identified as Fear Tactics
Police case filed against your name. Pay ₹5000 to stop it.,Scam,Fear Tactics,Identified as Fear Tactics
Your SIM card will be deactivated today. Verify immediately.,Scam,Fear Tactics,Identified as Fear Tactics
Login required: https://fakebank-login.com to secure your funds.,Scam,Phishing,Identified as Phishing
You have been selected for a lucky draw reward. Register here.,Scam,Reward Manipulation,Identified as Reward Manipulation
"Pre-approved loan of ₹5,00,000. Apply instantly with zero documents.",Scam,Loan Scam,Identified as Loan Scam
Get instant loan without credit check. Click here.,Scam,Loan Scam,Identified as Loan Scam
Your OTP is 123456. Share to verify your account.,Scam,OTP Fraud,Identified as OTP Fraud
Police case filed against your name. Pay ₹5000 to stop it.,Scam,Fear Tactics,Identified as Fear Tactics
Do not share OTP with anyone. Except us: [OTP],Scam,OTP Fraud,Identified as OTP Fraud
Government Grant Approved. Send your Aadhar number now.,Scam,Fake Authority,Identified as Fake Authority
"Pre-approved loan of ₹5,00,000. Apply instantly with zero documents.",Scam,Loan Scam,Identified as Loan Scam
Your OTP is 123456. Share to verify your account.,Scam,OTP Fraud,Identified as OTP Fraud
Government Grant Approved. Send your Aadhar number now.,Scam,Fake Authority,Identified as Fake Authority
Get instant loan without credit check. Click here.,Scam,Loan Scam,Identified as Loan Scam
"Pre-approved loan of ₹5,00,000. Apply instantly with zero documents.",Scam,Loan Scam,Identified as Loan Scam
We have restricted your UPI services. Update now.,Scam,Account Suspension,Identified as Account Suspension
Your Paytm account is suspended. Reactivate here.,Scam,Account Suspension,Identified as Account Suspension
We have restricted your UPI services. Update now.,Scam,Account Suspension,Identified as Account Suspension

In [ ]:
%%writefile test_scam_dataset.csv
text,label
Your Amazon order #123456 has been shipped,Scam
Can we reschedule our call to 2 PM instead of 1?,Not Scam
Get rich quick! Invest in our crypto scheme and double your money.,Scam
Can we reschedule our call to 2 PM instead of 1?,Not Scam
Urgent: Your bank account has been compromised. Verify your identity now.,Scam
Can we reschedule our call to 2 PM instead of 1?,Not Scam
Hi just checking in on the project status. Let me know if you need anything.,Not Scam
Meeting agenda for next week attached in this email.,Not Scam

In [ ]:
# OPTIONAL: upload scam_detection_dataset.csv (downloadable from the app) to evaluate on all rows
# from google.colab import files; files.upload()

## 3 · Key

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) ")
os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"   # or "gemini-3.5-flash-lite" if your key supports it

## 4 · Detect a single message

In [ ]:
import sys; sys.path.insert(0, "/content/scamguard_ai")
from pipeline.scam_detector.detector import ScamDetector
import json

detector = ScamDetector()          # strategy = "react" (see llm/prompts/react.md)
for msg in ["Congratulations! You've won a free iPhone. Click this link and enter your bank details to claim.",
            "Dear customer, your electricity bill of Rs 840 is due on the 5th. Pay on the official portal.",
            "Your package from Amazon is delayed. Track it here: bit.ly/track123"]:
    print(msg); print(json.dumps(detector.detect(msg), indent=2), "\n")

## 5 · See exactly which prompt is sent
Change the strategy by editing `PROMPT = load_prompt(...)` in `llm/prompts.py`; here we just print what the pipeline builds.

In [ ]:
from pipeline.scam_detector.builder import build_prompt
print(build_prompt("URGENT: Your account will be suspended! Call 555-FAKE immediately.", "react"))

## 6 · Evaluate on the sample (accuracy · recall · F1)
`--limit` keeps API calls low while you're on a free quota; remove it to use everything.

In [ ]:
!python evaluate.py scam_sample.csv --limit 20 --batch-size 5

### Where to go next
* Swap the prompt strategy (`fewshot.md`, `strict_json.txt`, `simplified.txt`) and compare F1.
* Validate every result with `llm/validator.py` (Pydantic).
* Add a retry when the parser returns “Uncertain”.